In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [2]:
from pathlib import Path

import polars as pl

from torch.utils.data import DataLoader
import lightning as L

from lightning.pytorch.profilers import PyTorchProfiler

from lightning.pytorch.loggers import TensorBoardLogger

from tabseq_torch.dl import (
    DataCatalog,
    LazyDataset,
    DDChunkReader,
    TargetsMetadata,
    random_split,
    cat_embedding_params,
    collate_padded_batch_fn,
    read_json,
)

from tabseq_torch.lit import PoolingType, LitImbBinTabLSTM

In [3]:
tx_path = Path(
    "/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/train_transactions_contest"
)
tx_path.exists()
dc = DataCatalog(
    src_tabts_path=tx_path,
    cat_path=tx_path.parent / "cat",
    meta_path=tx_path.parent / "meta",
    targets_path=tx_path.parent / "flag.parquet",
)
display(pl.scan_parquet(dc.src).collect_schema())
display(pl.scan_parquet(dc.targets).collect_schema())
dc.print_setnames()
encoded_columns_setname = "allcat"
display(pl.scan_parquet(dc.cat(encoded_columns_setname)).collect_schema())
cat_columns = list(read_json(dc.catnum_json(setname=encoded_columns_setname)).keys())
num_columns = ["amnt", "days_before", "transaction_number", "hour_diff"]

Schema([('app_id', Int32),
        ('amnt', Float64),
        ('currency', Int32),
        ('operation_kind', Int32),
        ('card_type', Int32),
        ('operation_type', Int32),
        ('operation_type_group', Int32),
        ('ecommerce_flag', Int32),
        ('payment_system', Int32),
        ('income_flag', Int32),
        ('mcc', Int32),
        ('country', Int32),
        ('city', Int32),
        ('mcc_category', Int32),
        ('day_of_week', Int32),
        ('hour', Int32),
        ('days_before', Int32),
        ('weekofyear', Int32),
        ('hour_diff', Int64),
        ('transaction_number', Int32),
        ('__index_level_0__', Int64)])

Schema([('app_id', Int64), ('flag', Int64)])

allcat
test


Schema([('app_id', Int32),
        ('currency', Int32),
        ('operation_kind', Int32),
        ('card_type', Int32),
        ('operation_type', Int32),
        ('operation_type_group', Int32),
        ('ecommerce_flag', Int32),
        ('payment_system', Int32),
        ('income_flag', Int32),
        ('mcc', Int32),
        ('country', Int32),
        ('city', Int32),
        ('mcc_category', Int32),
        ('day_of_week', Int32),
        ('hour', Int32),
        ('weekofyear', Int32)])

In [4]:
num_chunk_per_worker = 10
lazy_dataset = LazyDataset(
    chunk_reader=DDChunkReader(
        groupby="app_id",
        num_chunk_per_worker=num_chunk_per_worker,
        num_groups=TargetsMetadata.num_groups(dc.targets),
        memlimit_gb=0.2,
    )
)
lazy_dataset.set_catdata(
    data_path=dc.cat(encoded_columns_setname),
    columns_to_select=cat_columns,
    catnum_json=dc.catnum_json(setname=encoded_columns_setname),
)
lazy_dataset.set_numdata(
    data_path=dc.src,
    columns_to_select=num_columns,
)
lazy_dataset.set_targetmeta(dc.targets, target_column="flag")

In [5]:
TargetsMetadata.num_groups(dc.targets)

963811

In [6]:
test_size = 0.99
print(test_size)
train_ds, val_ds = random_split(lazy_dataset, test_size=test_size)
print(len(train_ds))
print(len(val_ds))
flag_counts = (
    pl.scan_parquet(dc.targets).select(pl.col("flag").unique_counts()).collect()
)
pos_weight = (flag_counts[1] / flag_counts[0]).item()
print(f"pos_weight={pos_weight}")
embedding_dims = lazy_dataset.embedding_dims_template(embedding_dim=20)
display(embedding_dims)
embedding_params = cat_embedding_params(
    catnum=read_json(dc.catnum_json(setname=encoded_columns_setname)),
    col_embedding_dims=embedding_dims,
)

0.99
9595
954216
pos_weight=0.02835684578237665


{'currency': 20,
 'operation_kind': 20,
 'card_type': 20,
 'operation_type': 20,
 'operation_type_group': 20,
 'ecommerce_flag': 20,
 'payment_system': 20,
 'income_flag': 20,
 'mcc': 20,
 'country': 20,
 'city': 20,
 'mcc_category': 20,
 'day_of_week': 20,
 'hour': 20,
 'weekofyear': 20}

In [7]:
b_lstm = LitImbBinTabLSTM(
    cat_enc_params=embedding_params,
    h_size=10,
    pos_weight=pos_weight,
    columns_by_type=lazy_dataset.type_columns,
    bidirectional=False,
    pooling_type=PoolingType.AVG,
    lr=1e-4,
    num_layers=1,
)

In [8]:
num_workers = 0
num_chunks = num_chunk_per_worker * num_workers
print(f"num num_chunks = {num_chunks} ~ XGB")
train_dl = DataLoader(
    dataset=train_ds,
    batch_size=20,
    collate_fn=collate_padded_batch_fn,
    num_workers=num_workers,
)
val_dl = DataLoader(
    dataset=val_ds,
    batch_size=20,
    collate_fn=collate_padded_batch_fn,
    num_workers=0,
)
print(len(train_dl))
print(len(val_dl))

num num_chunks = 0 ~ XGB
480
47711


In [9]:
%%time
print(f"test_size={test_size}")
print(f"num_workers={num_workers}")
print(f"num_chunk_per_worker={num_chunk_per_worker}")

logger = TensorBoardLogger(save_dir=tx_path.parent / "tb_logs", name="one_epoch")
print(f"log_dir={logger.log_dir}")

# profiler = PyTorchProfiler(
#     dirpath=tx_path.parent / "lit",
#     filename="profile_report.json",
#     export_to_chrome=True,
# )


trainer = L.Trainer(
    limit_val_batches=300,
    log_every_n_steps=150,
    val_check_interval=200,
    max_epochs=1,
    accelerator="auto",
    devices=1,
    logger=logger,
    enable_progress_bar=True,
)
print(f"tensorboard --host 0.0.0.0 --port 6006 --logdir={logger.log_dir}")
# trainer.fit(model=b_lstm, train_dataloaders=train_dl)
print(f"tensorboard --host 0.0.0.0 --port 6006 --logdir={logger.log_dir}")
trainer.fit(model=b_lstm, train_dataloaders=train_dl, val_dataloaders=val_dl)

/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/pr

test_size=0.99
num_workers=0
num_chunk_per_worker=10
log_dir=/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/tb_logs/one_epoch/version_13
tensorboard --host 0.0.0.0 --port 6006 --logdir=/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/tb_logs/one_epoch/version_13
tensorboard --host 0.0.0.0 --port 6006 --logdir=/home/jovyan/data/alpha/data_for_competition/train_transactions_contest/tb_logs/one_epoch/version_13


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/conda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/opt/conda/lib/python3.13/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


_validation_step_num_batchs_checksum = 2
40
40


...

/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Validation: |          | 0/? [00:00<?, ?it/s]

_validation_step_num_batchs_checksum = 602
6000
6000


/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/opt/conda/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


CPU times: user 10min 59s, sys: 2min 14s, total: 13min 14s
Wall time: 4min 15s
